# 22 Local Twitter Trend Normalization

Local-only deterministic normalization based on Phase 21 profiling findings.


**Notebook purpose:** Applies deterministic normalization rules (NFKC, case-fold, hashtag/ticker key variants) to the raw Twitter trending snapshot, producing normalized parquet outputs for downstream trend matching.

**Required data:** Twitter trending snapshot parquet + validation report + profiling findings doc (all produced by notebooks 01/02/21).

**Run order:** Run after notebook 21 (profiling). Run before notebook 23 (Bluesky text prep).

## 1. Load Phase Context And Local Inputs


In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover
    def display(value):
        print(value)


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'docs' / '22_local_twitter_trend_normalization.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root for Phase 22 notebook')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.trend_normalization import (
    normalize_twitter_trending_dataframe,
    summarize_duplicate_impact,
)

FULL_RAW_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_full.parquet'
SAMPLE_RAW_PARQUET_PATH = ROOT / 'data/samples/twitter_trending_sample_1000.parquet'
VALIDATION_REPORT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_validation_report.json'
PROFILE_FINDINGS_PATH = ROOT / 'docs/21_local_twitter_dataset_profiling_findings.md'
PROFILE_SUMMARY_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_profile_summary.json'

FULL_OUT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_normalized.parquet'
SAMPLE_OUT_PARQUET_PATH = ROOT / 'data/samples/twitter_trending_normalized_sample_1000.parquet'
SAMPLE_OUT_CSV_PATH = ROOT / 'data/samples/twitter_trending_normalized_sample_1000.csv'
NORMALIZATION_SUMMARY_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_normalization_summary.json'

required = [FULL_RAW_PATH, SAMPLE_RAW_PARQUET_PATH, VALIDATION_REPORT_PATH, PROFILE_FINDINGS_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required local Phase 22 inputs: ' + ', '.join(missing)
    )

validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding='utf-8'))
if validation_report.get('status') == 'fail' or validation_report.get('overall_pass') is False:
    raise RuntimeError('Validation report indicates fail; stop normalization phase.')

profile_summary = None
if PROFILE_SUMMARY_PATH.exists():
    profile_summary = json.loads(PROFILE_SUMMARY_PATH.read_text(encoding='utf-8'))

print('repo_root:', ROOT)
print('validation_status:', validation_report.get('status'))
print('profile_summary_loaded:', profile_summary is not None)

## 2. Load Raw Snapshot And Apply Normalization


In [ ]:
full_raw_df = pd.read_parquet(FULL_RAW_PATH)
sample_raw_df = pd.read_parquet(SAMPLE_RAW_PARQUET_PATH)

full_norm_df = normalize_twitter_trending_dataframe(full_raw_df)
sample_norm_df = normalize_twitter_trending_dataframe(sample_raw_df)

print('full_raw_rows:', len(full_raw_df))
print('full_norm_rows:', len(full_norm_df))
print('sample_raw_rows:', len(sample_raw_df))
print('sample_norm_rows:', len(sample_norm_df))


## 3. Exact Rules Implemented


In [ ]:
implemented_rules = [
    'Preserve raw columns (`name`, `date`, `num_hours`, `counts`) unchanged.',
    'Normalize trend text with Unicode NFKC and apostrophe normalization (`’`/`‘` -> `'"'"'`).',
    'Trim outer whitespace and collapse repeated inner whitespace.',
    'Case-fold to lowercase (`casefold`) for deterministic canonical keys.',
    'Normalize separators (`&`, `-`, `.`, `,`, `/`, `_`) to spaces.',
    'Remove non-word punctuation except hashtag and ticker markers (`#`, `$`) in cleaned text.',
    'Create `trend_name_clean` (with hashtag), `trend_name_clean_no_hash`, and `trend_name_clean_no_dollar`.',
    'Create `trend_name_alnum` helper key for alphanumeric-only comparison.',
    'Add helper metrics/flags: token count, char count, blank flag, hashtag flag, special-char flag, non-ASCII flag, URL-like flag.',
    'Preserve normalized date as `normalized_date` for downstream matching windows.',
]

for idx, rule in enumerate(implemented_rules, start=1):
    print(f"{idx}. {rule}")


## 4. Before/After Examples (Concise)


In [ ]:
example_cols = [
    'name',
    'trend_name_clean',
    'trend_name_clean_no_hash',
    'trend_name_clean_no_dollar',
    'trend_name_alnum',
    'is_blank_raw',
    'is_hashtag',
    'has_special_chars',
    'has_non_ascii',
    'has_url_like',
]

name_as_str = sample_norm_df['name'].astype(str)

masks = {
    'hashtags': name_as_str.str.strip().str.startswith('#'),
    'ticker_or_digit': name_as_str.str.contains(r'(?:\$|\d)', regex=True),
    'punctuation': name_as_str.str.contains(r'[^A-Za-z0-9#\s]', regex=True),
    'non_ascii': name_as_str.str.contains(r'[^\x00-\x7F]', regex=True),
    'multi_word': name_as_str.str.split().map(len) >= 2,
}

for label, mask in masks.items():
    print('\n' + label)
    display(sample_norm_df.loc[mask, example_cols].drop_duplicates().head(8))


## 5. Duplicate Impact Before vs After Normalization


In [ ]:
full_duplicate_impact = summarize_duplicate_impact(full_raw_df, full_norm_df)
sample_duplicate_impact = summarize_duplicate_impact(sample_raw_df, sample_norm_df)

pd.DataFrame([
    {
        'dataset': 'full',
        **{k: v for k, v in full_duplicate_impact.items() if k != 'collapsed_examples'},
    },
    {
        'dataset': 'sample_1000',
        **{k: v for k, v in sample_duplicate_impact.items() if k != 'collapsed_examples'},
    },
])


In [ ]:
collapsed_examples_df = pd.DataFrame(full_duplicate_impact['collapsed_examples'])
collapsed_examples_df.head(12)


## 6. Persist Local Normalized Outputs


In [ ]:
FULL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
SAMPLE_OUT_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

full_norm_df.to_parquet(FULL_OUT_PATH, index=False)
sample_norm_df.to_parquet(SAMPLE_OUT_PARQUET_PATH, index=False)
sample_norm_df.to_csv(SAMPLE_OUT_CSV_PATH, index=False)

print('wrote:', FULL_OUT_PATH)
print('wrote:', SAMPLE_OUT_PARQUET_PATH)
print('wrote:', SAMPLE_OUT_CSV_PATH)


## 7. Write Compact Normalization Summary (Optional)


In [ ]:
normalization_summary = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'inputs': {
        'full_raw_path': str(FULL_RAW_PATH),
        'sample_raw_parquet_path': str(SAMPLE_RAW_PARQUET_PATH),
        'validation_report_path': str(VALIDATION_REPORT_PATH),
        'profile_findings_path': str(PROFILE_FINDINGS_PATH),
    },
    'outputs': {
        'full_normalized_path': str(FULL_OUT_PATH),
        'sample_normalized_parquet_path': str(SAMPLE_OUT_PARQUET_PATH),
        'sample_normalized_csv_path': str(SAMPLE_OUT_CSV_PATH),
    },
    'shape': {
        'full_raw_rows': int(len(full_raw_df)),
        'full_normalized_rows': int(len(full_norm_df)),
        'sample_raw_rows': int(len(sample_raw_df)),
        'sample_normalized_rows': int(len(sample_norm_df)),
    },
    'duplicate_impact': {
        'full': full_duplicate_impact,
        'sample_1000': sample_duplicate_impact,
    },
    'implemented_rules': implemented_rules,
    'known_limitations': [
        'No fuzzy or semantic matching is performed in this phase.',
        'Alphanumeric helper keys may collapse non-Latin text and punctuation-rich variants; raw and cleaned keys are preserved for that reason.',
        'No joins to Bluesky data are performed in this phase.',
    ],
    'phase_ready': {
        'ready_for_phase_23_preparation': True,
        'reason': 'Deterministic normalization outputs exist with tested rules and duplicate-impact visibility.',
    },
}

NORMALIZATION_SUMMARY_PATH.write_text(json.dumps(normalization_summary, indent=2), encoding='utf-8')
print('wrote:', NORMALIZATION_SUMMARY_PATH)


## 8. Readiness Statement


In [ ]:
print('ready_for_phase_23_preparation:', normalization_summary['phase_ready']['ready_for_phase_23_preparation'])
print('reason:', normalization_summary['phase_ready']['reason'])
